# 🎯 Propensión a Compra — Gradient Boosting Classifier
> **Caso de negocio:** La Positiva Seguros · Campaña SOAT  
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC  
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

La Positiva contacta a **toda su cartera** para renovar el SOAT. El 68% de las llamadas del call center van a clientes con baja propensión, desperdiciando capacidad y presupuesto.

**Objetivo:** Predecir qué clientes comprarán SOAT en los próximos 15 días para priorizar el contacto comercial.

**KPIs de éxito:**
- Tasa de conversión +35% vs campaña masiva
- Reducir llamadas en frío -50%
- Top 20% clientes → 65% de las ventas

**Algoritmo:** Gradient Boosting Classifier (ensemble de árboles secuencial)

## 📦 Fase 0 — Instalación y setup

In [ ]:
# Instalación de librerías adicionales (Colab ya tiene scikit-learn, pandas, numpy)
!pip install -q shap plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
# Definición formal del problema
PROBLEMA = {
    'tipo': 'Clasificación binaria supervisada',
    'target': 'compro (0=no compró, 1=compró SOAT)',
    'ventana_prediccion': '15 días',
    'features': [
        'recencia_dias     → días desde última interacción con la marca',
        'n_compras_12m     → compras de seguros en últimos 12 meses',
        'engagement_score  → score de engagement digital (0-100)',
        'canal_digital     → si usa canales digitales (0/1)',
        'edad              → edad del cliente',
        'n_productos       → número de productos activos con La Positiva',
    ],
    'criterios_aceptacion': {
        'AUC-ROC': '>= 0.75',
        'Recall':  '>= 0.65  (minimizar falsos negativos)',
        'Precision': '>= 0.60',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con query a BigQuery o carga de CSV real
N = 1200

recencia     = np.random.randint(1, 121, N)
n_compras    = np.random.poisson(4, N).clip(0, 15)
engagement   = np.random.randint(10, 101, N)
canal_digital = (np.random.random(N) > 0.35).astype(int)
edad         = np.random.randint(22, 65, N)
n_productos  = np.random.randint(1, 6, N)

# DGP: proceso generador de datos con relaciones realistas
z = (-2.5
     - 0.025 * recencia      # más días inactivo → menos propensión
     + 0.22  * n_compras     # más historial de compras → más propensión
     + 0.035 * engagement    # más engagement → más propensión
     + 0.80  * canal_digital # cliente digital → más propensión
     + 0.30  * n_productos   # más productos → más vínculo con la marca
     + np.random.normal(0, 0.8, N))

prob  = 1 / (1 + np.exp(-z))
compro = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'cliente_id':      range(1, N+1),
    'recencia_dias':   recencia,
    'n_compras_12m':   n_compras,
    'engagement_score': engagement,
    'canal_digital':   canal_digital,
    'edad':            edad,
    'n_productos':     n_productos,
    'compro':          compro
})

print(f'Dataset: {df.shape}')
print(f'\nDistribución del target:')
print(df['compro'].value_counts(normalize=True).map('{:.1%}'.format))
df.head()

In [ ]:
# Estadísticas descriptivas por clase
print('=== COMPRARON (compro=1) ===')
display(df[df['compro']==1].describe().round(2))
print('\n=== NO COMPRARON (compro=0) ===')
display(df[df['compro']==0].describe().round(2))

In [ ]:
# Visualización: distribuciones por clase
features = ['recencia_dias', 'n_compras_12m', 'engagement_score', 'n_productos']
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f for f in features])

colors = {0: '#c0392b', 1: '#0d9488'}
for idx, feat in enumerate(features):
    r, c = idx // 2 + 1, idx % 2 + 1
    for clase, label in [(0, 'No compró'), (1, 'Compró')]:
        data = df[df['compro'] == clase][feat]
        fig.add_trace(
            go.Histogram(x=data, name=label, opacity=0.65,
                         marker_color=colors[clase],
                         showlegend=(idx == 0)),
            row=r, col=c
        )

fig.update_layout(barmode='overlay', height=500,
                  title='Distribución de features por clase del target',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de correlaciones
num_cols = ['recencia_dias', 'n_compras_12m', 'engagement_score',
            'canal_digital', 'edad', 'n_productos', 'compro']
corr = df[num_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Correlación entre variables')
fig.update_layout(height=450, paper_bgcolor='white')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['recencia_dias', 'n_compras_12m', 'engagement_score',
            'canal_digital', 'edad', 'n_productos']
TARGET   = 'compro'

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# Feature engineering
X['recencia_score'] = 1 / (X['recencia_dias'] + 1)           # inverso de recencia
X['rfm_proxy']      = X['n_compras_12m'] * X['engagement_score'] / 100  # frecuencia × engagement

print('Features finales:', X.columns.tolist())
print('Nulos:', X.isnull().sum().sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')
print(f'Tasa conversión train: {y_train.mean():.1%}')
print(f'Tasa conversión test:  {y_test.mean():.1%}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Entrenamiento del modelo GBM
# GBM: construye árboles secuencialmente, cada uno corrige los errores del anterior
gbm = GradientBoostingClassifier(
    n_estimators=150,    # número de árboles
    max_depth=4,         # profundidad máxima de cada árbol
    learning_rate=0.10,  # peso de cada árbol nuevo (shrinkage)
    subsample=0.8,       # fracción de datos para cada árbol (reduce overfitting)
    random_state=42
)

gbm.fit(X_train, y_train)
print('Modelo entrenado ✓')
print(f'Parámetros: {gbm.get_params()}')

In [ ]:
# Validación cruzada — estimación honesta del performance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(gbm, X_train, y_train, cv=cv, scoring='roc_auc')
cv_rec = cross_val_score(gbm, X_train, y_train, cv=cv, scoring='recall')

print(f'CV AUC-ROC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'CV Recall:  {cv_rec.mean():.3f} ± {cv_rec.std():.3f}')

In [ ]:
# Predicciones en test set
y_pred = gbm.predict(X_test)
y_prob = gbm.predict_proba(X_test)[:, 1]

# Importancia de variables
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': gbm.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(fi, x='importance', y='feature', orientation='h',
             title='Importancia de variables (GBM)',
             color='importance', color_continuous_scale='teal',
             text=fi['importance'].map('{:.3f}'.format))
fig.update_traces(textposition='outside')
fig.update_layout(height=380, yaxis={'categoryorder': 'total ascending'},
                  plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False)
fig.show()

## 5️⃣ Fase 5 — Evaluation

In [ ]:
# Métricas en test set
metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall':    recall_score(y_test, y_pred, zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred, zero_division=0),
    'AUC-ROC':   roc_auc_score(y_test, y_prob),
}

criterios = {'AUC-ROC': 0.75, 'Recall': 0.65, 'Precision': 0.60}

print('=== MÉTRICAS EN TEST SET ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral {umbral:.2f})'
    print(f'  {k:12s}: {v:.4f}  {estado}')

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                          name=f'GBM (AUC={auc:.3f})',
                          line=dict(color='#1a4c8c', width=2.5)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                          name='Random', line=dict(color='#ccc', dash='dash')))
fig.update_layout(title='Curva ROC — Propensión a Compra',
                  xaxis_title='Tasa Falsos Positivos', yaxis_title='Tasa Verdaderos Positivos',
                  height=400, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['No compró', 'Compró'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

print('\nInterpretación:')
tn, fp, fn, tp = cm.ravel()
print(f'  Verdaderos Positivos (VP): {tp} — detectados correctamente como compradores')
print(f'  Falsos Negativos    (FN): {fn} — compradores que NO detectamos (perdimos ventas)')
print(f'  Falsos Positivos    (FP): {fp} — contactamos en vano (costo innecesario)')
print(f'  Verdaderos Negativos(VN): {tn} — correctamente excluidos')

In [ ]:
# Análisis de lift: cuánto mejor es el modelo vs selección aleatoria
df_test = pd.DataFrame({'prob': y_prob, 'real': y_test.values})
df_test = df_test.sort_values('prob', ascending=False).reset_index(drop=True)

deciles = []
n = len(df_test)
for d in range(10):
    subset = df_test.iloc[d*n//10:(d+1)*n//10]
    conv_rate = subset['real'].mean()
    deciles.append({'decil': f'D{d+1}', 'tasa_conversion': conv_rate})

df_lift = pd.DataFrame(deciles)
baseline = df_test['real'].mean()
df_lift['lift'] = df_lift['tasa_conversion'] / baseline

fig = px.bar(df_lift, x='decil', y='lift',
             title=f'Lift por decil (baseline = {baseline:.1%})',
             text=df_lift['lift'].map('{:.2f}x'.format),
             color='lift', color_continuous_scale='teal')
fig.add_hline(y=1, line_dash='dash', line_color='red',
              annotation_text='Baseline (sin modelo)')
fig.update_traces(textposition='outside')
fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False)
fig.show()
print(f'\nEl top 20% de clientes por score concentra {df_lift["tasa_conversion"][:2].mean()/baseline:.1f}x más conversiones que el promedio')

In [ ]:
# SHAP values: interpretabilidad del modelo
explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Importancia global de features (SHAP)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm: dirección del impacto
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP — Dirección e intensidad del impacto por feature')
plt.tight_layout()
plt.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Exportar predicciones sobre todo el dataset
X_all = df[FEATURES].copy()
X_all['recencia_score'] = 1 / (X_all['recencia_dias'] + 1)
X_all['rfm_proxy']      = X_all['n_compras_12m'] * X_all['engagement_score'] / 100

df['score_propension'] = (gbm.predict_proba(X_all)[:, 1] * 100).round(1)
df['segmento'] = pd.cut(
    df['score_propension'],
    bins=[0, 35, 65, 100],
    labels=['Baja propensión', 'Media propensión', 'Alta propensión']
)

print('Distribución de segmentos:')
print(df['segmento'].value_counts())
print(f'\nClientes a contactar (Alta propensión): {(df["segmento"]=="Alta propensión").sum()}')
df[['cliente_id','score_propension','segmento']].sort_values('score_propension', ascending=False).head(10)

In [ ]:
# Guardar CSV para integración con CRM
df[['cliente_id','score_propension','segmento']].to_csv('scores_propension_soat.csv', index=False)
print('Archivo scores_propension_soat.csv guardado ✓')

# Guardar modelo
import joblib
joblib.dump(gbm, 'modelo_propension_soat.pkl')
print('Modelo modelo_propension_soat.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo de ROI
n_clientes = len(df)
n_alta = (df['segmento'] == 'Alta propensión').sum()
tasa_conv_modelo = df[df['segmento']=='Alta propensión']['compro'].mean()
tasa_conv_base   = df['compro'].mean()
costo_llamada    = 8  # S/. por llamada

ahorro_llamadas = (n_clientes - n_alta) * costo_llamada
lift_conversion  = tasa_conv_modelo / tasa_conv_base

print('=== RESUMEN EJECUTIVO ===')
print(f'Clientes en base total:              {n_clientes:,}')
print(f'Clientes a contactar (modelo):       {n_alta:,} ({n_alta/n_clientes:.0%})')
print(f'Tasa conversión modelo (Alta):       {tasa_conv_modelo:.1%}')
print(f'Tasa conversión campaña masiva:      {tasa_conv_base:.1%}')
print(f'Lift de conversión:                  {lift_conversion:.1f}x')
print(f'Ahorro en llamadas (S/.):            S/.{ahorro_llamadas:,.0f}')
print(f'\nArquitectura de producción:')
print('  Cloud Run API → score diario → CRM La Positiva → cola priorizada call center')